# Roslynator Parameter Count — Raw Output Extraction (C#)

This notebook analyzes **C# repositories** with **Roslynator** and derives **Parameter Count** by traversing the **Roslyn syntax tree** (`ParameterSyntax` nodes).

**Default benchmark repository:** [dotnet/runtime](https://github.com/dotnet/runtime)

> **Note:** **Parameter Count is a derived metric** for Roslynator. It is computed by counting `ParameterSyntax` nodes via Roslyn, while Roslynator diagnostics are preserved as raw evidence.


## Section 1 — Install Dependencies

Install Python packages, bootstrap .NET SDK, and install Roslynator CLI.


In [ ]:
!pip install -q pandas gitpython jupyter


In [ ]:
import os
import sys
from pathlib import Path

os.environ.pop('PYTHONPATH', None)
METRIC_ROOT = Path('.').resolve()
TOOL_ROOT = METRIC_ROOT / 'tool'
PROJECT_ROOT = METRIC_ROOT
for _ in range(8):
    if (PROJECT_ROOT / 'runtimes').is_dir() or (PROJECT_ROOT / 'README.md').is_file():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
RUNTIMES_ROOT = PROJECT_ROOT / 'runtimes'
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from run_parameter_count_benchmark_impl import (
    DOTNET_CHANNEL, build_parameter_count_analyzer, download_dotnet_sdk, dotnet_env, dotnet_executable,
    install_roslynator, run_command,
)

DOTNET_ROOT = download_dotnet_sdk(RUNTIMES_ROOT / 'dotnet-sdk', channel=DOTNET_CHANNEL)
ROSLYNATOR_PATH = install_roslynator(DOTNET_ROOT, RUNTIMES_ROOT / 'dotnet-tools')
ANALYZER_DLL = build_parameter_count_analyzer(DOTNET_ROOT)
version_stdout, version_stderr, _ = run_command([str(ROSLYNATOR_PATH), '--version'], env=dotnet_env(DOTNET_ROOT))
print((version_stdout or version_stderr).strip())
print(f'.NET SDK: {dotnet_executable(DOTNET_ROOT)}')
print(f'Roslynator: {ROSLYNATOR_PATH}')
print(f'ParameterCountAnalyzer: {ANALYZER_DLL}')


## Section 2 — Configuration


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/dotnet/runtime.git'

LOCAL_REPO_PATH = '/content/runtime'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/parameter_count_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path
import json
import sys

TOOL_ROOT = Path('tool').resolve()
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from _roslynator_parameter_count_notebook_utils import (
    NotebookLogger, compute_repository_stats, ensure_output_dir, preview_raw_output, resolve_repository_path,
)
from run_parameter_count_benchmark_impl import (
    analyze_targets, build_long_parameter_list, combine_xml_outputs, discover_csharp_files,
    discover_solution_and_project_files, dotnet_env, findings_to_json, merge_roslynator_results,
    parse_roslynator_text, parse_roslynator_xml, run_parameter_count_analyzer, run_roslynator_suite,
    save_csharp_inventory,
)
from IPython.display import display
import pandas as pd


## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL, repo_url=REPO_URL, local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH, if_clone_exists=IF_CLONE_EXISTS, logger=logger, clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

CS_FILES = discover_csharp_files(REPO_PATH)
if not CS_FILES:
    logger.error('No C# source files found in repository.', file=str(REPO_PATH))
    raise FileNotFoundError('No C# source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, CS_FILES)
logger.info(f'Repository ready at: {REPO_PATH}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (C# files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"C# files: {REPO_STATS['csharp_file_count']:,}")


## Section 5 — Discover C# Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'csharp_files_inventory.csv'
save_csharp_inventory(CS_FILES, INVENTORY_CSV)

print(f'Total C# Files Found: {len(CS_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Execute Roslynator

Run Roslynator in text, XML, and GitLab formats. Preserve stdout/stderr exactly as emitted.


In [ ]:
SOLUTIONS, PROJECTS = discover_solution_and_project_files(REPO_PATH)
TARGETS = analyze_targets(SOLUTIONS, PROJECTS)
ENV = dotnet_env(DOTNET_ROOT)
GITLAB_PATH = None

if not TARGETS:
    logger.error('No .sln or .csproj files discovered; Roslynator analysis skipped.')
    ROSLYNATOR_RAW = ''
    XML_PATHS = []
else:
    ROSLYNATOR_RAW, XML_PATHS, GITLAB_PATH = run_roslynator_suite(ROSLYNATOR_PATH, TARGETS, OUTPUT_PATH, ENV)
    logger.info(f'Roslynator analyzed {len(TARGETS)} target(s)')


## Section 7 — Raw Output Extraction


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'roslynator_raw_console_output.txt'
XML_PATH = OUTPUT_PATH / 'roslynator_output.xml'
JSON_PATH = OUTPUT_PATH / 'roslynator_output.json'

CONSOLE_PATH.write_text(ROSLYNATOR_RAW, encoding='utf-8')
combine_xml_outputs(XML_PATHS, XML_PATH)

FINDINGS_DF = merge_roslynator_results(
    parse_roslynator_xml([XML_PATH] if XML_PATH.exists() else XML_PATHS),
    parse_roslynator_text(ROSLYNATOR_RAW),
)
FINDINGS_CSV = OUTPUT_PATH / 'roslynator_findings.csv'
FINDINGS_DF.to_csv(FINDINGS_CSV, index=False)

JSON_PATH.write_text(json.dumps(findings_to_json(FINDINGS_DF, GITLAB_PATH), indent=2), encoding='utf-8')

logger.info(f'Parsed {len(FINDINGS_DF)} Roslynator findings')
preview_raw_output(ROSLYNATOR_RAW, RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 8 — Extract Parameter Count Using Roslyn Syntax API

Traverse C# source files and count `ParameterSyntax` nodes for methods, constructors, local functions, delegates, and record primary constructors.


In [ ]:
PARAM_CSV = OUTPUT_PATH / 'parameter_count.csv'
analyzer_stdout, analyzer_stderr, analyzer_code = run_parameter_count_analyzer(
    ANALYZER_DLL, REPO_PATH, PARAM_CSV, DOTNET_ROOT
)
if analyzer_stderr.strip():
    logger.error(analyzer_stderr.strip(), file='parameter_count_analyzer')
if analyzer_stdout.strip():
    CONSOLE_PATH.write_text(
        CONSOLE_PATH.read_text(encoding='utf-8') + '\n===== ParameterCountAnalyzer (stdout) =====\n' + analyzer_stdout,
        encoding='utf-8',
    )

PARAM_DF = pd.read_csv(PARAM_CSV) if PARAM_CSV.exists() else pd.DataFrame(
    columns=['file', 'namespace', 'class', 'method', 'line', 'parameter_count', 'parameter_names']
)
logger.info(f'Extracted parameter counts for {len(PARAM_DF)} members')
display(PARAM_DF.head())


## Section 9 — Parameter Count (Derived)

**Derived metric** from Roslyn syntax tree:

```text
Parameter_Count = Count(ParameterSyntax nodes)
```


In [ ]:
PARAM_VALUES = pd.to_numeric(PARAM_DF['parameter_count'], errors='coerce').dropna()
MAX_PARAM = int(PARAM_VALUES.max()) if not PARAM_VALUES.empty else 0
AVG_PARAM = round(float(PARAM_VALUES.mean()), 4) if not PARAM_VALUES.empty else 0.0

PARAM_SUMMARY_CSV = OUTPUT_PATH / 'parameter_count_summary.csv'
pd.DataFrame([{'metric_name': 'Parameter_Count', 'metric_value': MAX_PARAM}]).to_csv(PARAM_SUMMARY_CSV, index=False)

logger.info(f'Maximum Parameter Count (derived)={MAX_PARAM}')
display(pd.read_csv(PARAM_SUMMARY_CSV))


## Section 10 — Long Parameter List Detection

Flag methods where `Parameter_Count > 5`.


In [ ]:
LONG_PARAM_DF = build_long_parameter_list(PARAM_DF)
LONG_PARAM_CSV = OUTPUT_PATH / 'long_parameter_list.csv'
LONG_PARAM_DF.to_csv(LONG_PARAM_CSV, index=False)

LONG_PARAM_COUNT = int((LONG_PARAM_DF['status'] == 'Long Parameter List').sum())
logger.info(f'Long parameter list count={LONG_PARAM_COUNT}')
display(LONG_PARAM_DF)


## Section 11 — Summary Dashboard


In [ ]:
summary_df = pd.DataFrame([
    {'Metric': 'Total C# Files', 'Value': len(CS_FILES)},
    {'Metric': 'Total Methods', 'Value': len(PARAM_DF)},
    {'Metric': 'Average Parameter Count', 'Value': AVG_PARAM},
    {'Metric': 'Maximum Parameter Count', 'Value': MAX_PARAM},
    {'Metric': 'Long Parameter List Count', 'Value': LONG_PARAM_COUNT},
    {'Metric': 'Total Roslynator Diagnostics', 'Value': len(FINDINGS_DF)},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, XML_PATH, JSON_PATH, FINDINGS_CSV, PARAM_CSV, PARAM_SUMMARY_CSV,
    LONG_PARAM_CSV, INVENTORY_CSV, ERROR_LOG_PATH,
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 12 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 13 — Deliverables

```text
outputs/
├── roslynator_raw_console_output.txt
├── roslynator_output.xml
├── roslynator_output.json
├── roslynator_findings.csv
├── parameter_count.csv
├── parameter_count_summary.csv
├── long_parameter_list.csv
├── csharp_files_inventory.csv
└── error_log.txt
```
